In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

BACKGROUND = "#ffffff"
TEXT = "#111827"
TEXT_MUTED = "#6b7280"
GRID = "#e5e7eb"

COLOR = {
    "postgresql": "#00618a",
    "mysql": "#d97706",
    "python_postgresql": "#8a919e",
    "python_mysql": "#8a919e",
}

NAME = {
    "postgresql": "PostgreSQL",
    "mysql": "MySQL",
    "python_postgresql": "Python",
    "python_mysql": "Python",
}

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.weight": "bold",
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": TEXT,
    "ytick.color": TEXT,
    "figure.facecolor": BACKGROUND,
    "axes.facecolor": BACKGROUND,
    "savefig.facecolor": BACKGROUND,
    "axes.titleweight": "bold",
    "axes.titlecolor": TEXT,
    "figure.dpi": 110,
})

IMGS = "imgs"
os.makedirs(IMGS, exist_ok=True)


def save(figure, name):
    figure.savefig(os.path.join(IMGS, name), dpi=150, bbox_inches="tight")


def hide_spines(ax, keep=()):
    for side, spine in ax.spines.items():
        spine.set_visible(side in keep)
        if side in keep:
            spine.set_color(TEXT_MUTED)

In [ ]:
LOADS = ["1k", "10k", "100k"]

ROWS_PER_TABLE = {"1k": 1_000, "10k": 10_000, "100k": 100_000}

ENGINES = ["postgresql", "mysql"]

PYTHON_OF = {"postgresql": "python_postgresql", "mysql": "python_mysql"}

TIMES = pd.DataFrame(
    [
        ("insertion",    "postgresql",        226.006, 2252.994, 25784.750),
        ("query_1",      "postgresql",          2.928,    6.994,     52.796),
        ("query_2",      "postgresql",          8.393,   82.338,   1058.145),
        ("fetch_tables", "python_postgresql",  95.952,  152.862,   1222.129),
        ("query_1",      "python_postgresql",  30.413,  125.677,   1184.717),
        ("query_2",      "python_postgresql",  27.445,  108.340,    997.931),
        ("insertion",    "mysql",             370.492, 3596.290, 35933.864),
        ("query_1",      "mysql",               1.120,   12.144,    119.870),
        ("query_2",      "mysql",               7.281,  100.983,   1043.776),
        ("fetch_tables", "python_mysql",       82.083,  364.944,   3595.773),
        ("query_1",      "python_mysql",       48.183,  323.382,   3064.940),
        ("query_2",      "python_mysql",       40.885,  247.794,   2543.436),
    ],
    columns=["step", "engine", *LOADS],
)


def times_of(step, engine):
    row = TIMES[(TIMES["step"] == step) & (TIMES["engine"] == engine)]
    if row.empty:
        return np.full(len(LOADS), np.nan)
    return row[LOADS].to_numpy(dtype=float).ravel()


def format_ms(ms):
    if np.isnan(ms):
        return "—"
    if ms < 10:
        return f"{ms:.1f} ms"
    if ms < 1000:
        return f"{ms:.0f} ms"
    if ms < 60_000:
        return f"{ms / 1000:.1f} s"
    return f"{ms / 60_000:.1f} min"


def load_label(load):
    return f"{ROWS_PER_TABLE[load]:,} rows"


TIMES

In [ ]:
figure, ax = plt.subplots(figsize=(10, 5.5))

positions = np.arange(len(LOADS))
width = 0.72 / len(ENGINES)
top = max(np.nanmax(times_of("insertion", engine)) for engine in ENGINES)

for index, engine in enumerate(ENGINES):
    values = times_of("insertion", engine)
    offset = (index - (len(ENGINES) - 1) / 2) * width
    ax.bar(positions + offset, values, width * 0.84, color=COLOR[engine],
           label=NAME[engine], zorder=2)
    for x, value in zip(positions + offset, values):
        ax.text(x, value + top * 0.02, format_ms(value), ha="center", va="bottom",
                fontsize=13, color=TEXT)

ax.set_xticks(positions)
ax.set_xticklabels([load_label(load) for load in LOADS], fontsize=13)
ax.tick_params(axis="x", length=0, pad=12)
ax.set_yticks([])
ax.set_xlim(-0.6, len(LOADS) - 0.4)
ax.set_ylim(0, top * 1.18)
hide_spines(ax, keep=("top",))

ax.legend(loc="upper left", frameon=False, fontsize=12)
ax.set_title("Insertion time by load size", fontsize=18, pad=22)
figure.tight_layout()
save(figure, "insertion.png")
plt.show()

In [ ]:
figure, ax = plt.subplots(figsize=(10, 6))

positions = np.arange(len(LOADS))
DASH = {"query_1": (0, (5, 2)), "query_2": "solid"}

for engine in ENGINES:
    for step in ("query_1", "query_2"):
        values = times_of(step, engine)
        ax.plot(positions, values, color=COLOR[engine], linewidth=2,
                linestyle=DASH[step], marker="o", markersize=8,
                markerfacecolor=COLOR[engine], markeredgecolor=BACKGROUND,
                markeredgewidth=2,
                label=f"{NAME[engine]} · {step.replace('_', ' ')}", zorder=3)
        ax.text(positions[-1] + 0.06, values[-1], f"  {format_ms(values[-1])}",
                color=TEXT, fontsize=12, va="center")

ax.set_yscale("log")
ax.set_xticks(positions)
ax.set_xticklabels([load_label(load) for load in LOADS], fontsize=13)
ax.set_xlim(-0.25, len(LOADS) - 0.3)
ax.tick_params(axis="y", labelsize=11, colors=TEXT_MUTED)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
hide_spines(ax, keep=("left", "bottom"))

ax.legend(loc="upper left", frameon=False, fontsize=11, labelcolor=TEXT, ncol=2,
          handlelength=4.2, handletextpad=0.8, columnspacing=2.2, markerscale=0.9)
ax.set_title("SQL query times", fontsize=18, pad=22)
figure.tight_layout()
save(figure, "sql_queries.png")
plt.show()

In [ ]:
def compare_engines(step):
    figure, ax = plt.subplots(figsize=(9.5, 5.5))

    positions = np.arange(len(LOADS))
    width = 0.72 / len(ENGINES)
    by_engine = {engine: times_of(step, engine) for engine in ENGINES}
    top = max(np.nanmax(values) for values in by_engine.values())

    for index, engine in enumerate(ENGINES):
        offset = (index - (len(ENGINES) - 1) / 2) * width
        ax.bar(positions + offset, by_engine[engine], width * 0.84,
               color=COLOR[engine], label=NAME[engine], zorder=2)
        for x, value in zip(positions + offset, by_engine[engine]):
            ax.text(x, value + top * 0.02, format_ms(value), ha="center",
                    va="bottom", fontsize=12, color=TEXT)

    ax.set_xticks(positions)
    ax.set_xticklabels([load_label(load) for load in LOADS], fontsize=13)
    ax.tick_params(axis="x", length=0, pad=12)
    ax.set_yticks([])
    ax.set_xlim(-0.6, len(LOADS) - 0.4)
    ax.set_ylim(0, top * 1.18)
    hide_spines(ax, keep=("top",))

    ax.legend(loc="upper left", frameon=False, fontsize=12)
    ax.set_title(f"{step.replace('_', ' ').capitalize()} — PostgreSQL vs MySQL",
                 fontsize=18, pad=22)
    figure.tight_layout()
    save(figure, f"engines_{step}.png")
    plt.show()


for step in ("query_1", "query_2"):
    compare_engines(step)

In [ ]:
def compare_with_python(step, engine):
    in_python = PYTHON_OF[engine]
    figure, ax = plt.subplots(figsize=(9.5, 5.5))

    positions = np.arange(len(LOADS))
    series = [(engine, times_of(step, engine)),
              (in_python, times_of(step, in_python))]
    width = 0.72 / len(series)
    top = max(np.nanmax(values) for _, values in series)

    for index, (key, values) in enumerate(series):
        offset = (index - (len(series) - 1) / 2) * width
        ax.bar(positions + offset, values, width * 0.84, color=COLOR[key],
               label=NAME[key], zorder=2)
        for x, value in zip(positions + offset, values):
            ax.text(x, value + top * 0.02, format_ms(value), ha="center",
                    va="bottom", fontsize=12, color=TEXT)

    ax.set_xticks(positions)
    ax.set_xticklabels([load_label(load) for load in LOADS], fontsize=13)
    ax.tick_params(axis="x", length=0, pad=12)
    ax.set_yticks([])
    ax.set_xlim(-0.6, len(LOADS) - 0.4)
    ax.set_ylim(0, top * 1.18)
    hide_spines(ax, keep=("top",))

    ax.legend(loc="upper left", frameon=False, fontsize=12)
    ax.set_title(f"{step.replace('_', ' ').capitalize()} — {NAME[engine]} vs Python",
                 fontsize=18, pad=22)
    figure.tight_layout()
    save(figure, f"{step}_{engine}_vs_python.png")
    plt.show()


for engine in ENGINES:
    for step in ("query_1", "query_2"):
        compare_with_python(step, engine)

In [ ]:
OVERALL_SERIES = [
    ("postgresql", "PostgreSQL", None),
    ("mysql", "MySQL", None),
    ("python_postgresql", "Python · from PostgreSQL", None),
    ("python_mysql", "Python · from MySQL", "///"),
]

totals = {key: times_of("query_1", key) + times_of("query_2", key)
          for key, _, _ in OVERALL_SERIES}

figure, panels = plt.subplots(1, len(LOADS), figsize=(13, 4.4))

positions = np.arange(len(OVERALL_SERIES))[::-1]

for panel, (ax, load) in enumerate(zip(panels, LOADS)):
    values = [totals[key][panel] for key, _, _ in OVERALL_SERIES]
    top = max(values)

    for position, (key, _, hatch), value in zip(positions, OVERALL_SERIES, values):
        ax.barh(position, value, 0.62, color=COLOR[key], hatch=hatch,
                edgecolor=BACKGROUND, linewidth=1.2, zorder=2)
        ax.text(value + top * 0.04, position, format_ms(value), va="center",
                fontsize=12, color=TEXT)

    ax.set_xlim(0, top * 1.42)
    ax.set_ylim(-0.7, len(OVERALL_SERIES) - 0.3)
    ax.set_xticks([])
    ax.set_yticks(positions if panel == 0 else [])
    if panel == 0:
        ax.set_yticklabels([label for _, label, _ in OVERALL_SERIES], fontsize=12)
    ax.tick_params(axis="y", length=0, pad=10)
    hide_spines(ax)
    ax.set_title(load_label(load), fontsize=14, pad=14, color=TEXT_MUTED)

figure.suptitle("Both queries summed", fontsize=18, fontweight="bold",
                color=TEXT, y=1.06)
figure.tight_layout(w_pad=3)
save(figure, "overall.png")
plt.show()